# SurviveCity v1 — extended Kaggle T4 retrain

The original Colab run was **12 GRPO steps in 3h53m**, which gave a clean
1.7× reward / 2.0× episode-length lift but only 2 logged training points
(steps 10 and 12). This notebook **trains from base for 20 steps** and saves
every 4, so the next eval pass yields:

  - a 5-point training curve (steps 4, 8, 12, 16, 20) instead of 2 points
  - a cross-checkpoint eval trend (the curve in §5.3 of the report)
  - tighter confidence intervals via larger `n_t` / `n_b`

Time budget on Kaggle T4: ~19 min/step × 20 = **~6.3 hours of training** plus
~30 min eval. Fits well inside Kaggle's 12h session cap.

**Repo isolation: this notebook pushes to `noanya/zombiee-v1-extended`,
not `noanya/zombiee`.** The original step-12 adapter, checkpoints, TB run dir,
and eval JSON on `noanya/zombiee` are NOT modified by this run. If the new
numbers turn out worse or broken, fall back to the original — no recovery
needed. If they're better, repoint v1.tex's URLs to the new repo and ship.


## 0. Bind to a single GPU BEFORE any imports

Kaggle ships dual T4 by default; bnb + DataParallel breaks CUBLAS in that
combo. Setting CUDA_VISIBLE_DEVICES=0 here (before torch is imported anywhere)
makes torch see exactly one device.


In [ ]:
import os, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("python:", sys.version.split()[0])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


## 1. Pull HF + GitHub tokens (Kaggle Secrets → env)

**HF_TOKEN** is required (the Hub push will fail without it).
**GITHUB_TOKEN** is only required if the GitHub repo is private; public clone
works without it.


In [ ]:
HF_TOKEN = None
GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    try: HF_TOKEN = _sec.get_secret("HF_TOKEN")
    except Exception: pass
    try: GITHUB_TOKEN = _sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
GITHUB_TOKEN = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN in Kaggle Secrets (Add-ons → Secrets) before running."
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN     set ({len(HF_TOKEN)} chars, prefix={HF_TOKEN[:6]}...)")
print(f"GITHUB_TOKEN {'set ('+str(len(GITHUB_TOKEN))+' chars)' if GITHUB_TOKEN else 'absent (ok if repo is public)'}")


## 2. Pin torch cu121 FIRST so accelerate can't downgrade it

In [ ]:
import subprocess
def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print("STDERR:", r.stderr[-800:])
    if check and r.returncode != 0: raise RuntimeError(f"pip failed (rc={r.returncode})")
    return r

if "torch" in sys.modules:
    raise RuntimeError("torch already imported — restart kernel via Run → Factory reset.")

pip("install", "-q",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "--upgrade-strategy=only-if-needed",
    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121", "torchaudio==2.5.1+cu121")


## 3. Install pinned dep stack (no-deps + force-reinstall)

`--no-deps + --force-reinstall` guarantees our pins take effect even on
Kaggle's pre-populated dist-packages (the Achilles heel of the v2 run).
The base image already has compatible tokenizers/safetensors/numpy.


In [ ]:
pip("install", "-q", "--no-deps", "--force-reinstall",
    "transformers==4.46.3",
    "peft==0.13.2",
    "trl==0.15.2",
    "bitsandbytes==0.43.3",
    "accelerate==1.1.1",
    "datasets==3.1.0")
pip("install", "-q", "--no-deps", "mergekit", check=False)
pip("install", "-q",
    "huggingface_hub>=0.26", "tensorboard", "tbparse", "psutil", "pydantic")
pip("uninstall", "-y", "torchao", check=False)

# transformers 4.46 removed TRANSFORMERS_CACHE; some optional imports still use it.
import transformers.utils.hub as _hub
if not hasattr(_hub, "TRANSFORMERS_CACHE"):
    _hub.TRANSFORMERS_CACHE = getattr(_hub, "HF_HUB_CACHE", None) or os.path.expanduser("~/.cache/huggingface/hub")


## 4. Verify pin took + torch sees the GPU

In [ ]:
import importlib.metadata as _m
expected = {"transformers":"4.46.3","peft":"0.13.2","trl":"0.15.2",
            "bitsandbytes":"0.43.3","accelerate":"1.1.1","datasets":"3.1.0"}
bad = []
for pkg, want in expected.items():
    try:
        got = _m.version(pkg)
        ok = got == want
        print(f"  {pkg:15s} want={want:10s} got={got:10s} {'OK' if ok else 'MISMATCH'}")
        if not ok: bad.append((pkg, want, got))
    except _m.PackageNotFoundError:
        print(f"  {pkg:15s} NOT INSTALLED"); bad.append((pkg, want, "missing"))
assert not bad, f"pinned versions didn't take: {bad}"

import torch
print(f"\n  torch={torch.__version__}")
print(f"  cuda available: {torch.cuda.is_available()}  device count: {torch.cuda.device_count()}")
print(f"  torch built with cuda: {torch.version.cuda}")
print(f"  CUDA_VISIBLE_DEVICES env: {os.environ.get('CUDA_VISIBLE_DEVICES','(unset)')}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available. Most common causes on Kaggle:\n"
        "  1) Notebook's accelerator is OFF. Fix: Settings (right sidebar) → "
        "Accelerator → 'GPU T4 x1' or 'GPU P100', then restart kernel.\n"
        "  2) torch.version.cuda is None → CPU-only torch was installed. "
        "Re-run cell 2 (the cu121 pin) and ensure cell 3 didn't pull a CPU torch.\n"
        f"     This kernel reports torch.version.cuda = {torch.version.cuda!r}.\n"
        "  3) Weekly GPU quota exhausted on this Kaggle account. Check the "
        "'Your Account' page for remaining hours."
    )
print(f"  GPU: {torch.cuda.get_device_name(0)} cc={torch.cuda.get_device_capability(0)}")
free, total = torch.cuda.mem_get_info(0)
print(f"  VRAM free={free/1e9:.1f}/{total/1e9:.1f} GB")

from transformers.utils.import_utils import is_bitsandbytes_available
assert is_bitsandbytes_available(), "bnb compat broken; check the pin install logs above."
print("  bnb compat OK")


## 5. Clone repo + cd into v1

Uses `GIT_TERMINAL_PROMPT=0` + `GIT_ASKPASS=/bin/echo` so a missing token
fails fast instead of hanging waiting for a username. If the GitHub repo is
private, `GITHUB_TOKEN` (from cell 1) is embedded via `http.extraheader` —
this avoids baking the token into the cloned repo's `.git/config`.

If clone still fails, the most likely cause is the repo being private
without GITHUB_TOKEN set in Kaggle Secrets, or a typo in the URL.


In [ ]:
import base64
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["GIT_ASKPASS"] = "/bin/echo"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp"
REPO_DIR = os.path.join(WORK, "zombiee")
REPO_URL = os.environ.get("REPO_URL", "https://github.com/SirjanSingh/zombiee.git")

if os.path.isdir(REPO_DIR):
    print(f"repo already cloned at {REPO_DIR} — pulling latest")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    cmd = ["git", "clone", "--depth=1", REPO_URL, REPO_DIR]
    if GITHUB_TOKEN:
        # http.extraheader keeps the token out of the resulting .git/config
        auth = "Authorization: Basic " + base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode()
        ).decode()
        cmd = ["git", "-c", f"http.extraheader={auth}"] + cmd[1:]
        print(f"cloning {REPO_URL} (with GITHUB_TOKEN auth)...")
    else:
        print(f"cloning {REPO_URL} (no GITHUB_TOKEN; only works if repo is public)...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(
            f"git clone failed (rc={r.returncode}). Likely causes:\n"
            f"  1) Repo is private and GITHUB_TOKEN isn't set. Add it to "
            f"Kaggle Secrets (Add-ons → Secrets) with key 'GITHUB_TOKEN'.\n"
            f"  2) Typo in REPO_URL ({REPO_URL}). Override via env var.\n"
            f"  3) Token doesn't have repo read access. Generate a new "
            f"fine-grained token with 'Contents: Read' on the target repo."
        )
V1 = os.path.join(REPO_DIR, "v1")
assert os.path.isdir(V1), f"v1/ not found at {V1} (repo may have a different layout)"
os.chdir(V1)
sys.path.insert(0, V1)
print("CWD:", os.getcwd())


## 6. Pre-create / login to the Hub repo

`HUB_REPO` is INTENTIONALLY a separate repo from `noanya/zombiee`. The
original step-12 artefacts must remain untouched — they're what the report's
existing figures and Table 2 are derived from. If this extended run lands
cleanly, repoint `report/v1/v1.tex`'s URLs to the new repo. If it doesn't,
nothing on `noanya/zombiee` changed; just abandon this run.


In [ ]:
# Override via env var if you want a different name; the default is
# isolated from the original `noanya/zombiee`.
HUB_REPO = os.environ.get("HUB_REPO", "noanya/zombiee-v1-extended")
ORIGINAL_REPO = "noanya/zombiee"
assert HUB_REPO != ORIGINAL_REPO, (
    f"HUB_REPO must NOT equal {ORIGINAL_REPO} — that would overwrite the "
    f"step-12 root adapter. Pick a different name (default is "
    f"'noanya/zombiee-v1-extended')."
)
print(f"  training will push to: https://huggingface.co/{HUB_REPO}")
print(f"  original repo (untouched): https://huggingface.co/{ORIGINAL_REPO}")

from huggingface_hub import HfApi, login
login(token=HF_TOKEN, add_to_git_credential=False)
HfApi().create_repo(repo_id=HUB_REPO, repo_type="model", exist_ok=True, private=False)
print(f"  hub repo ready.")


## 7. Train — 20 steps from base, save every 4

Hyperparameters (deliberately the SAME as the original run for clean
comparability — the only thing changing is `--max-steps 12 → 20`):

  - `--max-steps 20`         (was 12)
  - `--save-steps 4`         (5 checkpoints: 4, 8, 12, 16, 20)
  - `--save-total-limit 5`
  - `--num-generations 4`    (unchanged)
  - `--lr 1e-5`              (unchanged)
  - `--temperature 0.9`      (unchanged)
  - `--beta 0.04`            (unchanged)
  - `--lora-r 16`            (unchanged)

Expected: ~16 min/step × 20 = ~5.3h on T4. Hub pushes happen on every save.


In [ ]:
import importlib, training.train as _train_mod
importlib.reload(_train_mod)

ARGS = [
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--max-steps", "20",
    "--save-steps", "4",
    "--save-total-limit", "5",
    "--num-generations", "4",
    "--lr", "1e-5",
    "--temperature", "0.9",
    "--beta", "0.04",
    "--lora-r", "16",
    "--lora-alpha", "32",
    "--seed", "42",
    "--output-dir", os.path.join(WORK, "lora_v1_extend"),
    "--push-to-hub",
    "--hub-model-id", HUB_REPO,
]
sys.argv = ["train.py"] + ARGS
print("Launching training:")
for a in ARGS: print("  ", a)
print()
_train_mod.main()


## 8. Training summary — TB curve across the 5 logged points

In [ ]:
from tbparse import SummaryReader
import pandas as pd, glob

ckpt_dir = os.path.join(WORK, "lora_v1_extend")
runs = sorted(glob.glob(os.path.join(ckpt_dir, "runs", "*")))
for r in runs[-2:]:
    print(f"\n=== {os.path.basename(r)} ===")
    df = SummaryReader(r, pivot=True).scalars
    if df is None or df.empty:
        print("  (empty)"); continue
    df = df.sort_values("step").reset_index(drop=True)
    cols = ["step"] + [c for c in [
        "train/loss", "train/reward", "train/reward_std", "train/kl",
        "train/grad_norm", "train/learning_rate"
    ] if c in df.columns]
    print(df[cols].to_string(index=False))


## 9. Eval — n=30 baseline + n=20 trained, generate plots

Uses v1's eval.py. Output: `eval_step_0020.json` (or whatever the final step
was), `eval_step_0020_bars.png`, `eval_history.png`. Drop those into
`report/v1/figures/` and update Table 2 in v1.tex.


In [ ]:
import importlib, training.eval as _eval_mod
importlib.reload(_eval_mod)

# v1 eval.py uses a single --episodes total (split half/half). 50 → 25/25.
# Bumped to 60 → 30 baseline + 30 trained for tighter CIs.
EVAL_ARGS = [
    "--lora-path", os.path.join(WORK, "lora_v1_extend"),
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--episodes", "60",
    "--seed", "42",
    "--plots-dir", os.path.join(WORK, "lora_v1_extend", "plots"),
]
sys.argv = ["eval.py"] + EVAL_ARGS
print("Launching eval:")
for a in EVAL_ARGS: print("  ", a)
print()
try:
    _eval_mod.main()
except SystemExit:
    pass


## 10. Push the eval artefacts to the Hub repo

So `report/v1/v1.tex` can pull them by URL or `huggingface_hub.snapshot_download`.
Pushed to the EXTENDED repo, not the original.


In [ ]:
from huggingface_hub import upload_folder

eval_dir = os.path.join(WORK, "lora_v1_extend", "plots")
if os.path.isdir(eval_dir):
    upload_folder(
        folder_path=eval_dir,
        path_in_repo="eval_results",
        repo_id=HUB_REPO,
        repo_type="model",
        commit_message="extended-training eval (step 20, n_t=30 vs n_b=30)",
    )
    print(f"  uploaded {eval_dir} -> {HUB_REPO}/eval_results")
    print(f"  view: https://huggingface.co/{HUB_REPO}/tree/main/eval_results")
else:
    print(f"  WARN: no plots dir at {eval_dir} — eval may not have run.")


## 11. Print headline numbers to paste into v1.tex Table 2

The previous step-12 numbers in the report:

| metric | step-12 baseline (n=30) | step-12 trained (n=10) |
|---|---|---|
| Survival rate | 0.0% (0/30) | 10.0% (1/10) |
| Mean total reward | 0.457 | 0.797 ± 0.41 |
| Mean episode length | 19.1 ± 7.3 | 37.6 ± 22.1 |
| JSON parse-success | 100% (random) | 100% (0 fails) |

Replace those rows with the numbers printed below.


In [ ]:
import json, glob
eval_jsons = sorted(glob.glob(os.path.join(WORK, "lora_v1_extend", "plots", "*.json")))
if not eval_jsons:
    print("no eval json found")
else:
    latest = eval_jsons[-1]
    print(f"=== {os.path.basename(latest)} ===")
    with open(latest) as f:
        d = json.load(f)
    print(json.dumps({
        "baseline": d.get("baseline_summary"),
        "trained":  d.get("trained_summary"),
        "delta":    d.get("delta"),
    }, indent=2))
